# 00 - Simulate your own DECAL samples (primer)

This notebook is the entry point to the pipeline: it shows how to generate your own
detector samples from scratch with `ddsim`, then hand them to notebooks 01-03 to study how
the reconstruction performs. The intended path:

1. Reproduce the baseline sample (a photon gun into the Si-W ECal barrel).
2. Vary the **gun** (particle type, energy) and the **geometry** (pixel pitch, silicon
   depth) to make your own samples.
3. Feed those samples to **01 -> 02 -> 03** and see how the energy reconstruction changes.

The *simulation* steps run in an EAF **terminal** (they call `ddsim`); this notebook is
the guide, plus a final cell to sanity-check a sample you produced.

## 1. The moving pieces

| Piece | File | Role |
|-------|------|------|
| Simulation driver | `ddsim` (DD4hep/Geant4) | runs the Geant4 simulation |
| Top-level geometry | [`geometry/SiD_TestBeam.xml`](../geometry/SiD_TestBeam.xml) | detector + global constants (incl. **pixel pitch**) |
| DECAL definition | [`geometry/my_custom_ecal.xml`](../geometry/my_custom_ecal.xml) | the Si-W ECal layers (incl. **silicon depth**) |
| Steering | [`sim/run_sim.py`](../sim/run_sim.py) | the particle **gun**, physics list, output |
| Output | `*.root` (EDM4hep) | what notebooks 01-03 read with `uproot` |

## 2. Set up the environment (EAF terminal)

In a JupyterLab terminal on EAF (one-time: symlink the launcher into your home per README / handbook §6.3), then source it:

```bash
source ~/setup_calomaps.sh     # Key4hep + the OpenGL shim; cd's into sim/
```

> **One gotcha**: steering files
> must be **plain ASCII**. A single fancy character (an em-dash `-` typed as `—`, or a
> "smart quote") in a comment lands in the run metadata and makes the EDM4hep writer abort
> before any event is written. Stick to `-` and straight quotes `'` `"`.

## 3. Generate your first sample (a few events)

From the `geometry/` directory (so the compact file's includes resolve), point `ddsim` at
the top-level geometry and the steering, and write **EDM4hep** `.root` (what 01-03 read):

```bash
cd $CALOMAPS_HOME/geometry
mkdir -p $CALOMAPS_DATA_BASE/my_first          # ddsim/ROOT won't create the output dir for you
ddsim --compactFile SiD_TestBeam.xml \
      --steeringFile ../sim/run_sim.py \
      -N 10 \
      --outputFile $CALOMAPS_DATA_BASE/my_first/photons.root
```

A handful of events takes under a minute. `run_sim.py` derives a default output name from
the particle, but always pass `--outputFile NAME.root` so you control the path and get the
EDM4hep ROOT the analysis notebooks expect. Then jump to the **verify** cell at the bottom
to inspect it.

## 4. The knobs you can turn

### (a) Particle & energy - environment variables (no file edits)
`run_sim.py` reads the gun from environment variables, so you switch particle or energy
without editing any file:

| Variable | Default | Meaning |
|----------|---------|---------|
| `CALOMAPS_GUN_PARTICLE` | `gamma` | particle name (`pi+`, `pi-`, `proton`, ...) |
| `CALOMAPS_GUN_PMIN_GEV` | `5.0`   | momentum-spectrum minimum (GeV) |
| `CALOMAPS_GUN_PMAX_GEV` | `400.0` | momentum-spectrum maximum (GeV) |
| `CALOMAPS_GUN_ENERGY_GEV` | (unset) | if set, a mono-energetic beam (overrides PMIN/PMAX) |

```bash
# Photons, 5-400 GeV (the default):
ddsim --compactFile SiD_TestBeam.xml --steeringFile ../sim/run_sim.py -N 10 \
      --outputFile $CALOMAPS_DATA_BASE/my_first/photons.root

# Pi+, same spectrum - just set one variable, no file edits:
CALOMAPS_GUN_PARTICLE=pi+ ddsim --compactFile SiD_TestBeam.xml \
      --steeringFile ../sim/run_sim.py -N 10 \
      --outputFile $CALOMAPS_DATA_BASE/my_first/pions.root

# A single 50 GeV pi-:
CALOMAPS_GUN_PARTICLE=pi- CALOMAPS_GUN_ENERGY_GEV=50 ddsim --compactFile SiD_TestBeam.xml \
      --steeringFile ../sim/run_sim.py -N 1 --outputFile $CALOMAPS_DATA_BASE/my_first/pim50.root
```
- **gamma** -> electromagnetic showers (pair-production + bremsstrahlung).
- **pi+/pi-/proton** -> hadronic showers; comparing the EM and hadronic responses is
  a core thread of notebooks 01-03.
- The gun fires a +y "pencil beam" (`thetaMin=thetaMax=90`, `phiMin=phiMax=90`).

### (b) Pixel pitch - [`geometry/SiD_TestBeam.xml`](../geometry/SiD_TestBeam.xml)
```xml
<constant name="ECal_cell_size" value="0.1*mm" />   <!-- 100 um pixels (default) -->
```
This sets the readout `CartesianGridXY` segmentation. Try `0.05*mm` (finer) or `0.5*mm`
(coarser) to study how granularity affects clustering and reconstruction. (Commented
alternatives sit right next to it in the file.)

### (c) Silicon depth - [`geometry/my_custom_ecal.xml`](../geometry/my_custom_ecal.xml)
```xml
<slice material="Silicon" thickness="0.032*cm" sensitive="yes" .../>   <!-- 320 um -->
```
This is the active sensor thickness. **It appears in both `<layer>` blocks** (the first 20
layers and the next 10) - change **both** to keep the detector consistent. (For context, the
tungsten absorber is `0.25*cm` in the first 20 layers and `0.5*cm` in the last 10.)

After any geometry edit, just re-run the `ddsim` command in section 3 - the new geometry is
picked up automatically.

## 5. Generate a full dataset (many events)

For real studies you want thousands of events. [`sim/generate_batched.sh`](../sim/generate_batched.sh)
(or [`sim/generate_dataset.sh`](../sim/generate_dataset.sh), a simpler 200x100 variant) fans
out parallel `ddsim` jobs. They resolve the geometry + steering by absolute path, so you can
run them from anywhere, and they take the **same** particle variable - no file edits:

```bash
bash $CALOMAPS_HOME/sim/generate_batched.sh                          # photons
CALOMAPS_GUN_PARTICLE=pi+ bash $CALOMAPS_HOME/sim/generate_batched.sh   # pi+
```

Each run writes its own dataset directory under `$CALOMAPS_DATA_BASE/`: the photon default
is `data_spectrum_100um_400GeV/` (unchanged), and a non-photon run gets a `_<particle>`
suffix, e.g. `data_spectrum_100um_400GeV_piplus/`. Inside are the `.root` files notebook 02
globs over. Size the run with the `CALOMAPS_NJOBS` / `CALOMAPS_NEVENTS` environment variables
(e.g. `CALOMAPS_NJOBS=40 CALOMAPS_NEVENTS=50 bash $CALOMAPS_HOME/sim/generate_batched.sh`).

## 6. Then: study the reconstruction (notebooks 01 -> 02 -> 03)

Point the analysis notebooks at the samples **you** generated:

- **01_detector_and_data** - geometry + a first look at one of your events.
- **02_data_extraction** - extract your `.root` dataset into a compact `.npz`
  (set the dataset path/name near the top to your `DATASET_NAME`).
- **03_ml_training_and_eval** - train the quantile ensembles and view the Neyman dashboard.

Change one knob at a time (e.g. pitch 100 um -> 50 um, or gamma -> pi+) and watch how the
reconstruction resolution responds.

## 7. Verify a sample you produced

The cell below opens an EDM4hep `.root` with `uproot` and prints a quick summary. Set
`SAMPLE` to a file you generated (section 3). Works for any particle - it reads the primary
PDG and energy from the file rather than assuming photons. **Kernel:** `Key4hep (CPU)` (registered by `setup_calomaps.sh`; reload the JupyterLab tab if the picker doesn't list it yet). Do **not** pick the generic `Python 3 (ipykernel)` — that base `/opt/conda` Python has no `uproot`.

In [1]:
import os
import numpy as np
import uproot

# Point this at a file you generated, e.g. f"{os.environ['CALOMAPS_DATA_BASE']}/my_first/photons.root"
_data_base = os.environ.get("CALOMAPS_DATA_BASE", os.path.expanduser("~/CALOMAPS-data"))
_cands = [
    os.path.join(_data_base, "my_first", "photons.root"),
    os.path.join(_data_base, "my_first", "pions.root"),
    os.path.join(_data_base, "data_spectrum_100um_400GeV", "sim_photons_part1.root"),
]
SAMPLE = next((p for p in _cands if p and os.path.exists(p)), None)
if SAMPLE is None:
    raise FileNotFoundError("Set SAMPLE to a .root file you generated (see section 3).")

f = uproot.open(SAMPLE)
t = f["events"]
print(f"sample: {SAMPLE}")
print(f"events in file: {t.num_entries}")

a = t.arrays(["MCParticles.PDG", "MCParticles.momentum.x", "MCParticles.momentum.y",
              "MCParticles.momentum.z", "MCParticles.mass", "ECalBarrelHits.energy"])
ev = 0
pdg0 = a["MCParticles.PDG"][ev][0]
px = a["MCParticles.momentum.x"][ev][0]; py = a["MCParticles.momentum.y"][ev][0]
pz = a["MCParticles.momentum.z"][ev][0]; m = a["MCParticles.mass"][ev][0]
# NOTE: EDM4hep stores momentum / mass / hit energy in GeV (no MeV->GeV scaling needed).
Eprim = np.sqrt(px**2 + py**2 + pz**2 + m**2)
edep = a["ECalBarrelHits.energy"][ev].to_numpy().sum()
print(f"\nevent 0: primary PDG={pdg0}, E={Eprim:.1f} GeV")
print(f"event 0: MCParticles stored = {len(a['MCParticles.PDG'][ev])}  (primary + kept truth)")
print(f"event 0: ECalBarrel hits    = {len(a['ECalBarrelHits.energy'][ev])}")
print(f"event 0: deposited in Si     = {edep:.2f} GeV  (sampling fraction ~{edep/Eprim:.2%})")

# incident-energy spectrum across the file (this is the label notebooks 02/03 reconstruct)
import awkward as ak
Eall = np.sqrt(a["MCParticles.momentum.x"][:, 0]**2 + a["MCParticles.momentum.y"][:, 0]**2
               + a["MCParticles.momentum.z"][:, 0]**2 + a["MCParticles.mass"][:, 0]**2)
print(f"\nprimary energy across {t.num_entries} events: "
      f"{ak.min(Eall):.1f} - {ak.max(Eall):.1f} GeV (mean {ak.mean(Eall):.1f})")

sample: /nashome/m/murtazas/CALOMAPS/models/primer_demo_photons.root
events in file: 20



event 0: primary PDG=22, E=330.9 GeV
event 0: MCParticles stored = 1  (primary + kept truth)
event 0: ECalBarrel hits    = 42012
event 0: deposited in Si     = 4.94 GeV  (sampling fraction ~1.49%)

primary energy across 20 events: 44.6 - 330.9 GeV (mean 210.3)
